In [14]:
import os
import sys
import time
import requests
import shutil
from urllib.parse import quote_plus
import xml.etree.ElementTree as ET
from tqdm import tqdm
import zstandard as zstd
import hashlib

In [4]:
BASE_URL = "https://checkpoints.mainnet.sui.io"
SAVE_DIR = "sui_checkpoints"

In [34]:
def download_checkpoint(seq_num):

    fname = f"{seq_num}.chk"                # filename to fetch
    url = f"{BASE_URL}/{fname}"               # full URL
    local_path = os.path.join(SAVE_DIR, fname)  # save path

    try:
        r = requests.get(url, stream=True)
        if r.status_code != 200:
            print(f"[{seq_num}.chk] Not found or end reached.")
            

        # Stream download to file
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=64*1024):
                if chunk:
                    f.write(chunk)

        print(f"[{seq_num}.chk] Downloaded successfully.")

    except Exception as e:
        print(f"[{seq_num}.chk] Error: {e}")

    return local_path


def basic_parse_checkpoint(path):
    """Parse minimal metadata from checkpoint file."""
    data = open(path, "rb").read()

    return {
        "file": path,
        "size_bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest()
    }


def download_sequence(end=100000011):
    """Download a range of checkpoints."""
    results = []

    initial_checkpoints = [0, 1, 10, 100, 1000, 10000, 100000, 1000000, 10000000, 100000000]

    for checkpoint in initial_checkpoints:
        print(f"Downloading checkpoint {checkpoint}…")
        path = download_checkpoint(checkpoint)
        if not path:
            break
        
        info = basic_parse_checkpoint(path)
        print(f"Parsed: {info}")
        results.append(info)
    

    for i in range(100000000, end + 1):
        print(f"Downloading checkpoint {i}…")
        path = download_checkpoint(i)
        if not path:
            break

        info = basic_parse_checkpoint(path)
        print(f"Parsed: {info}")
        results.append(info)

        # Handle trailing zeros
        i_str = str(i)
        while i_str.endswith('0') and len(i_str) > 1:
            i_str = i_str[:-1]  # remove one trailing zero
            j = int(i_str)
            print(f"Also downloading checkpoint {j} (from trailing zero rule)…")
            path_j = download_checkpoint(j)
            if path_j:
                info_j = basic_parse_checkpoint(path_j)
                print(f"Parsed: {info_j}")
                results.append(info_j)

    return results



In [35]:
download_sequence()

[0.chk] Downloaded successfully.
Parsed: {'file': 'sui_checkpoints\\0.chk', 'size_bytes': 642945, 'sha256': '6e134e3bfa54aac09c05139b5cb0518551ac7de1709c46d365b0fd82c10f48d8'}
[1.chk] Downloaded successfully.
Parsed: {'file': 'sui_checkpoints\\1.chk', 'size_bytes': 1479, 'sha256': 'b8b638c7a23e7fcc717e6d5c7d7f7c4ff770a34ee526e4ce40fcad48aa295b15'}
[10.chk] Downloaded successfully.
Parsed: {'file': 'sui_checkpoints\\10.chk', 'size_bytes': 1483, 'sha256': '84f497140c5e0541c602099748d2c3ca74485bef70d87e529b9d08a5e5db6eaa'}
[100.chk] Downloaded successfully.
Parsed: {'file': 'sui_checkpoints\\100.chk', 'size_bytes': 1483, 'sha256': 'c6eb005fc9b7605d3457788b8fe963173ea1f22611e8cfd94bbef94a67416e10'}
[1000.chk] Downloaded successfully.
Parsed: {'file': 'sui_checkpoints\\1000.chk', 'size_bytes': 1502, 'sha256': 'd7fcbf86071dfd30e098bb3d747563df2d313d1f6573b9aefada78271beba8c3'}
[10000.chk] Downloaded successfully.
Parsed: {'file': 'sui_checkpoints\\10000.chk', 'size_bytes': 1498, 'sha256': '3

[{'file': 'sui_checkpoints\\0.chk',
  'size_bytes': 642945,
  'sha256': '6e134e3bfa54aac09c05139b5cb0518551ac7de1709c46d365b0fd82c10f48d8'},
 {'file': 'sui_checkpoints\\1.chk',
  'size_bytes': 1479,
  'sha256': 'b8b638c7a23e7fcc717e6d5c7d7f7c4ff770a34ee526e4ce40fcad48aa295b15'},
 {'file': 'sui_checkpoints\\10.chk',
  'size_bytes': 1483,
  'sha256': '84f497140c5e0541c602099748d2c3ca74485bef70d87e529b9d08a5e5db6eaa'},
 {'file': 'sui_checkpoints\\100.chk',
  'size_bytes': 1483,
  'sha256': 'c6eb005fc9b7605d3457788b8fe963173ea1f22611e8cfd94bbef94a67416e10'},
 {'file': 'sui_checkpoints\\1000.chk',
  'size_bytes': 1502,
  'sha256': 'd7fcbf86071dfd30e098bb3d747563df2d313d1f6573b9aefada78271beba8c3'},
 {'file': 'sui_checkpoints\\10000.chk',
  'size_bytes': 1498,
  'sha256': '3ef7c957bf0c3bb928ec538b2fb4884695a8fcb752e6fba2b7f983db650be790'},
 {'file': 'sui_checkpoints\\100000.chk',
  'size_bytes': 1500,
  'sha256': '5d8a34d15274031417ca4b76fef582843b4be4c76f5c28ef3b31479d20418eb1'},
 {'file': 